In [1]:
%pip install -qU langchain langchain-core langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.4 MB/s eta 0:00:00


In [2]:
import os

try:
    from google.colab import userdata
    groq_api_key = userdata.get("GROQ_API_KEY") or userdata.get("GROQ_API_KEY")
except Exception:
    groq_api_key = None

if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key
    print("Groq API key configured successfully.")
else:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

Groq API key configured successfully.


In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=300
)

In [ ]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

def chat(message, history):
    messages = [
        SystemMessage(
            content="You are a helpful AI assistant. Answer clearly and practically."
        )
    ]

    # Gradio message format:
    # [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")

        if role == "user":
            messages.append(HumanMessage(content=content))
        elif role == "assistant":
            messages.append(AIMessage(content=content))

    # Add current user message
    messages.append(HumanMessage(content=message))

    response = chat_model.invoke(messages)
    return response.content


# --- Custom theme ---
theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    body_background_fill="*neutral_50",
    block_background_fill="white",
    block_border_width="1px",
    block_shadow="*shadow_drop_lg",
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_700",
    button_primary_text_color="white",
    input_background_fill="white",
)

# --- Custom CSS for extra polish ---
custom_css = """
#header {
    text-align: center;
    padding: 12px 0 4px 0;
}
#header h1 {
    font-size: 1.8rem;
    font-weight: 700;
    margin-bottom: 4px;
}
#header p {
    color: #6b7280;
    font-size: 0.95rem;
}
.gradio-container {
    max-width: 900px !important;
    margin: auto !important;
}
footer {visibility: hidden}
#chatbot {
    border-radius: 16px !important;
}
"""

with gr.Blocks(title="Groq Chatbot") as demo:

    with gr.Column(elem_id="header"):
        gr.Markdown(
            """
            # 🤖 Groq Chatbot
            <p>Powered by LangChain + GPT-OSS-120B — full conversation context included</p>
            """
        )

    chatbot = gr.Chatbot(
        elem_id="chatbot",
        height=500,
        avatar_images=(None, "https://groq.com/wp-content/uploads/2024/03/favicon.png"),
    )

    gr.ChatInterface(
        fn=chat,
        chatbot=chatbot,
        examples=[
            "Explain quantum computing in simple terms",
            "Write a Python function to reverse a string",
            "Give me 3 tips for better sleep",
        ],
        cache_examples=False,
    )

    gr.Markdown(
        "<p style='text-align:center; color:#9ca3af; font-size:0.8rem; margin-top:10px;'>"
        "Built with Gradio · LangChain · Groq</p>"
    )

demo.launch(share=True, debug=True, theme=theme, css=custom_css)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fd6d24943d165e6cb8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
